# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the Kilifi County mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}\nDescription: {metadata_dict['description']}")
print("\nPublished:", metadata_dict['datePublished'])
print("Dataset ID:", metadata_dict['identifier'])
print("Authors (@id):", [a['@id'] for a in metadata_dict['author']])

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema references each entity by its `@id`, which includes record sets, fields, and columns. Listing these IDs helps identify key dataset elements.

In [ ]:
# List record sets by @id
record_sets_metadata = dataset.metadata.to_json().get('recordSet', [])

if not record_sets_metadata:
    # Try extracting record sets from dataset.records() (mlcroissant auto-detects)
    # List available record sets and their IDs
    print("No explicit record sets found in metadata. Attempting to enumerate available sets from the dataset...")
    try:
        all_record_sets = dataset.record_sets()
        print("Record sets detected:")
        for rs in all_record_sets:
            print(f"  - @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, fields: {[f['@id'] for f in rs.get('field', [])]}")
    except Exception as e:
        print("Could not enumerate record sets automatically:", e)
else:
    print("Record sets listed in schema:")
    for rs_meta in record_sets_metadata:
        print(f"  - @id: {rs_meta['@id']}, name: {rs_meta.get('name', 'N/A')}")
        # List field/column IDs
        field_list = rs_meta.get('field', [])
        for fld in field_list:
            print(f"    - field @id: {fld['@id']} name: {fld.get('name', 'N/A')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> **Note:** Each record set, field, and column is referenced by its `@id` to ensure consistency and transparency.

Below, we extract all available record sets using their `@id`s and load them into DataFrames for further analysis.

In [ ]:
record_sets = []
record_sets_ids = []
try:
    # Attempt to get all record sets and their @id
    for rs in dataset.record_sets():
        record_sets.append(rs)
        record_sets_ids.append(rs['@id'])
except Exception as e:
    print("Could not enumerate record sets automatically:", e)

if not record_sets_ids:
    print("No record sets found. Please consult the schema or contact the dataset provider.")
else:
    print("RecordSet @id values:")
    for rid in record_sets_ids:
        print(rid)

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in record set {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field (e.g., GAD-7 score) and demonstrate filtering, normalization, and grouping by a categorical field (e.g., gender or education level). All fields are referenced by their `@id`.

> If the specific field `@id` is unknown, please check the column outputs from section 3.

In [ ]:
# Replace these IDs and field names with those found in your dataset
record_set_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes.get(record_set_id, pd.DataFrame())

# Example field @id for GAD-7 score (update if field is named differently in your dataset)
numeric_field = 'GAD_7_score'  # Example, replace with actual @id or column name
# Example group field (e.g., gender; update to actual @id/column name)
group_field = 'gender'  # Example, replace with actual @id or column name

# Make sure the numeric field exists
if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if present
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field}' not found in columns: {df.columns.tolist()}")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, an example histogram of GAD-7 scores, using the appropriate field `@id` or column name as determined from EDA.

In [ ]:
# Visualize GAD-7 score distribution
if numeric_field in df.columns and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8,4))
    df[numeric_field].dropna().hist(bins=10)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
else:
    print(f"Field '{numeric_field}' not present or all null.")

## 6. Conclusion

- Loaded the Kilifi County mental health survey dataset using `mlcroissant`, referencing all entities by their `@id`.
- Explored available record sets and fields, extracted specific data for analysis.
- Performed EDA with filtering, normalization, and grouping demonstrated on numeric and categorical fields.
- Visualized key distributions to highlight dataset trends.

Further analyses may include more advanced statistics, correlation studies, and predictive modeling depending on the research question and available fields.